# Amazon ML Challenge — visible, reproducible Kaggle training run

This is the complete experiment launcher, not a black-box wrapper. While it runs, it shows:

- the exact Git branch and commit used;
- every required TSV discovered under the Kaggle dataset mount;
- file sizes, schemas, accelerator details, and run configuration;
- the preprocessing, candidate-generation, feature, training, inference, and validation stages;
- progress for large TSV chunks and every XGBoost tuning trial;
- validation and untouched holdout **macro F0.5**, selected threshold, candidate recall, and output files.

The challenge data remains TSV. The pipeline streams it in chunks and creates temporary Parquet partitions under `/kaggle/temp` because repeatedly scanning giant TSV files during training would be wasteful.

## 1. Run controls

Before starting on Kaggle, select **T4 x2** and enable Internet for the first run. The default branch is `main`. Set environment variables only when you intentionally want a different repository, branch, team name, or attached E5 model snapshot.

In [1]:
import json
import os
import queue
import shutil
import subprocess
import sys
import tempfile
import threading
import time
from datetime import datetime
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
RUN_FULL_PIPELINE = IS_KAGGLE and os.environ.get('BER_RUN_FULL_PIPELINE', '1') == '1'
REPO_URL = os.environ.get('BER_REPO_URL', 'https://github.com/saltypal/ASR_AmazonML.git')
GIT_BRANCH = os.environ.get('BER_GIT_BRANCH', 'main')
TEAM_NAME = os.environ.get('BER_TEAM_NAME', 'team_name')
KAGGLE_DATASET_MOUNT = Path(os.environ.get(
    'BER_KAGGLE_DATASET_PATH',
    '/kaggle/input/datasets/satyapaladugu/amazon-ml-challenge',
))

def banner(title: str) -> None:
    print('\n' + '=' * 96)
    print(title)
    print('=' * 96, flush=True)

banner('RUN CONTROLS')
print(f'Environment        : {"Kaggle" if IS_KAGGLE else "Local validation"}')
print(f'Run full pipeline  : {RUN_FULL_PIPELINE}')
print(f'Repository         : {REPO_URL}')
print(f'Latest branch      : {GIT_BRANCH}')
print(f'Preferred data path: {KAGGLE_DATASET_MOUNT}')
print(f'Team name          : {TEAM_NAME}')


RUN CONTROLS


Environment        : Local validation
Run full pipeline  : False
Repository         : https://github.com/saltypal/ASR_AmazonML.git
Latest branch      : main
Preferred data path: \kaggle\input\datasets\satyapaladugu\amazon-ml-challenge
Team name          : team_name


## 2. Delete the temporary checkout and fetch the latest branch

Every Kaggle run deletes only `/kaggle/working/amazonml-src`, clones the repository again, explicitly fetches the latest remote branch tip, and checks out that exact commit. It does not delete `/kaggle/input`, the attached dataset, or final files elsewhere in `/kaggle/working`.

In [2]:
def locate_local_repo(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'code' / 'business_entity_resolution' / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the AmazonML repository.')

if IS_KAGGLE:
    PROJECT_DIR = Path('/kaggle/working/amazonml-src')
    if PROJECT_DIR.parent != Path('/kaggle/working'):
        raise RuntimeError(f'Refusing to reset unexpected path: {PROJECT_DIR}')

    banner('CLEAN GIT SYNC')
    print(f'[git] Removing old temporary checkout: {PROJECT_DIR}', flush=True)
    if PROJECT_DIR.is_symlink():
        PROJECT_DIR.unlink()
    elif PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)

    print(f'[git] Cloning: {REPO_URL}', flush=True)
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(PROJECT_DIR)], check=True)
    print(f'[git] Fetching latest origin/{GIT_BRANCH}', flush=True)
    subprocess.run(
        ['git', 'fetch', '--prune', 'origin', f'refs/heads/{GIT_BRANCH}:refs/remotes/origin/{GIT_BRANCH}'],
        cwd=PROJECT_DIR,
        check=True,
    )
    subprocess.run(
        ['git', 'checkout', '--force', '--detach', f'refs/remotes/origin/{GIT_BRANCH}'],
        cwd=PROJECT_DIR,
        check=True,
    )
else:
    PROJECT_DIR = locate_local_repo(Path.cwd().resolve())

PACKAGE_ROOT = PROJECT_DIR / 'code' / 'business_entity_resolution'
for module_name in list(sys.modules):
    if module_name == 'ber' or module_name.startswith('ber.'):
        del sys.modules[module_name]
sys.path.insert(0, str(PACKAGE_ROOT / 'src'))

if IS_KAGGLE:
    print('[setup] Installing the freshly fetched project package.', flush=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-e', str(PACKAGE_ROOT)],
        check=True,
    )

GIT_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR, text=True
).strip()
print(f'\nProject directory: {PROJECT_DIR}')
print(f'Resolved commit  : {GIT_COMMIT}')


Project directory: D:\Bunker\BaseCamp\AmazonML
Resolved commit  : 1ae5b8dbbaff7ecff1e2ec02b1363d25dac97ac3


## 3. Verify hardware and reproducibility

The two GPUs are used as independent workers for multilingual E5 encoding. XGBoost trains on one CUDA device because it operates on one candidate-pair table; wrapping it in PyTorch `DataParallel` would not make XGBoost use two GPUs.

In [3]:
import numpy as np
import pandas as pd
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

hardware = {
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu_count': torch.cuda.device_count(),
    'gpu_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}
banner('HARDWARE')
print(json.dumps(hardware, indent=2))
if RUN_FULL_PIPELINE and torch.cuda.device_count() < 2:
    raise RuntimeError('This full configuration expects Kaggle T4 x2. Change the accelerator and restart.')


HARDWARE


{
  "python": "3.14.0",
  "pandas": "3.0.2",
  "torch": "2.11.0+cu126",
  "cuda_available": false,
  "gpu_count": 0,
  "gpu_names": []
}


## 4. Discover and validate every challenge TSV

The preferred mount is exactly `/kaggle/input/datasets/satyapaladugu/amazon-ml-challenge/`. Kaggle sometimes exposes the same dataset as `/kaggle/input/amazon-ml-challenge/`, so the resolver checks that compatibility path next. It accepts a directory only when all seven required files exist with the expected structure.

In [4]:
EXPECTED_RELATIVE_FILES = [
    Path('train/train_source1.tsv'),
    Path('train/train_source2.tsv'),
    Path('train/train_source3.tsv'),
    Path('train/train_ground_truth.tsv'),
    Path('test/test_source1.tsv'),
    Path('test/test_source2.tsv'),
    Path('test/test_source3.tsv'),
]

def contains_complete_dataset(root: Path) -> bool:
    return all((root / relative_path).is_file() for relative_path in EXPECTED_RELATIVE_FILES)

def resolve_data_root() -> Path:
    search_roots = []
    if IS_KAGGLE:
        search_roots.extend([
            KAGGLE_DATASET_MOUNT,
            Path('/kaggle/input/amazon-ml-challenge'),
        ])
    search_roots.append(PROJECT_DIR / 'dataset')

    checked = []
    for mount in search_roots:
        checked.append(str(mount))
        if contains_complete_dataset(mount):
            return mount
        if mount.is_dir():
            for source1_file in mount.rglob('train_source1.tsv'):
                candidate = source1_file.parent.parent
                if contains_complete_dataset(candidate):
                    return candidate
    raise FileNotFoundError(
        'Could not find all seven required TSV files. Checked:\n- ' + '\n- '.join(checked)
    )

DATA_ROOT = resolve_data_root()
from ber.io import validate_input_headers

input_inventory = validate_input_headers(DATA_ROOT)
inventory_rows = []
for key, details in input_inventory.items():
    path = Path(details['path'])
    inventory_rows.append({
        'dataset_key': key,
        'file': path.name,
        'size_GiB': round(details['bytes'] / (1024**3), 3),
        'columns': ', '.join(details['columns']),
        'path': str(path),
    })
inventory_table = pd.DataFrame(inventory_rows)

banner('DATASET INVENTORY — ALL REQUIRED FILES FOUND')
print(f'Resolved dataset root: {DATA_ROOT}')
print(f'Required files found : {len(inventory_table)}/{len(EXPECTED_RELATIVE_FILES)}')
print(f'Total input size     : {inventory_table["size_GiB"].sum():.3f} GiB')
display(inventory_table)


DATASET INVENTORY — ALL REQUIRED FILES FOUND


Resolved dataset root: D:\Bunker\BaseCamp\AmazonML\dataset
Required files found : 7/7
Total input size     : 2.347 GiB


,dataset_key,file,size_GiB,columns,path
0,train_source1,train_source1.tsv,0.196,"entity_id, business_name, business_address, co...",D:\Bunker\BaseCamp\AmazonML\dataset\train\trai...
1,train_source2,train_source2.tsv,0.456,"entity_id, business_name, business_address, co...",D:\Bunker\BaseCamp\AmazonML\dataset\train\trai...
2,train_source3,train_source3.tsv,0.469,"entity_id, business_name, business_address, co...",D:\Bunker\BaseCamp\AmazonML\dataset\train\trai...
3,train_ground_truth,train_ground_truth.tsv,0.118,"source1_entity_id, matched_entity_ids",D:\Bunker\BaseCamp\AmazonML\dataset\train\trai...
4,test_source1,test_source1.tsv,0.163,"entity_id, business_name, business_address, co...",D:\Bunker\BaseCamp\AmazonML\dataset\test\test_...
5,test_source2,test_source2.tsv,0.474,"entity_id, business_name, business_address, co...",D:\Bunker\BaseCamp\AmazonML\dataset\test\test_...
6,test_source3,test_source3.tsv,0.471,"entity_id, business_name, business_address, co...",D:\Bunker\BaseCamp\AmazonML\dataset\test\test_...


## 5. What algorithm is actually running?

This is a supervised multilingual **entity-resolution** pipeline:

1. **Unicode NFKC + casefold normalization** makes compatible Unicode forms and letter case comparable without translating names or addresses into English.
2. **Candidate generation** avoids comparing every row with every other row. It combines exact blocking, character n-gram TF-IDF retrieval, and bounded multilingual semantic reranking.
3. **Multilingual E5-small** runs only on a small cross-script candidate lane. It helps connect spellings written in different scripts; it does not replace the full model.
4. **Pair features** measure name, address, digit, token, country, retrieval, and semantic agreement for each candidate pair.
5. **GPU XGBoost** classifies candidate pairs. A bounded random hyperparameter search evaluates each trial on entity-level validation groups.
6. **Threshold tuning** directly maximizes macro F0.5, the precision-heavy challenge metric.
7. A separate untouched **holdout split** estimates generalization after model and threshold selection.

The notebook reports F0.5 rather than ordinary accuracy because accuracy would be misleading when negative candidate pairs greatly outnumber real matches.

## 6. Load the eight-hour configuration and multilingual model

The run is bounded to 480 minutes, keeps a final output reserve, and limits both the number and total wall time of tuning trials. The model revision is pinned for reproducibility and licensing review.

In [5]:
import yaml

MODEL_ID = 'intfloat/multilingual-e5-small'
MODEL_REVISION = 'fd1525a9fd15316a2d503bf26ab031a61d056e98'
E5_MODEL_PATH = os.environ.get('BER_E5_MODEL_PATH')

if RUN_FULL_PIPELINE and not E5_MODEL_PATH:
    attached_models = [
        path.parent
        for path in Path('/kaggle/input').rglob('modules.json')
        if (path.parent / 'config.json').is_file()
    ]
    if attached_models:
        E5_MODEL_PATH = str(attached_models[0])
        print(f'[model] Using attached model snapshot: {E5_MODEL_PATH}', flush=True)
    else:
        from huggingface_hub import snapshot_download
        print(f'[model] Downloading pinned {MODEL_ID} revision {MODEL_REVISION}', flush=True)
        E5_MODEL_PATH = snapshot_download(
            MODEL_ID,
            revision=MODEL_REVISION,
            local_dir='/kaggle/temp/multilingual-e5-small',
        )

BASE_CONFIG = PACKAGE_ROOT / 'configs' / ('kaggle_t4x2.yaml' if IS_KAGGLE else 'smoke.yaml')
runtime_config = yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
if E5_MODEL_PATH:
    runtime_config.setdefault('embeddings', {})['model_path'] = E5_MODEL_PATH

RUNTIME_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path(tempfile.mkdtemp(prefix='ber-notebook-'))
RUNTIME_CONFIG = RUNTIME_ROOT / 'ber_runtime_config.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(runtime_config, sort_keys=False), encoding='utf-8')
WORK_DIR = Path('/kaggle/temp/ber-run') if IS_KAGGLE else RUNTIME_ROOT / 'work'
OUTPUT_DIR = Path('/kaggle/working/submission/output') if IS_KAGGLE else RUNTIME_ROOT / 'output'

algorithm_summary = {
    'normalization': 'Unicode NFKC + casefold; no English translation',
    'candidate_retrieval': 'exact blocks + character n-gram TF-IDF + cross-script E5 reranking',
    'semantic_model': MODEL_ID,
    'classifier': 'XGBoost binary:logistic with histogram trees',
    'training_device': runtime_config['training']['device'],
    'tuning_trials': runtime_config['training']['tuning_trials'],
    'tuning_timeout_minutes': runtime_config['training']['tuning_timeout_seconds'] / 60,
    'early_stopping_rounds': runtime_config['training']['early_stopping_rounds'],
    'maximum_boost_rounds': runtime_config['training']['max_boost_rounds'],
    'selection_metric': 'entity-level macro F0.5',
    'time_budget_minutes': runtime_config['project']['time_budget_minutes'],
    'output_reserve_minutes': runtime_config['project']['output_reserve_minutes'],
}
banner('ALGORITHM AND RUN CONFIGURATION')
print(json.dumps(algorithm_summary, indent=2))
print('\nFull resolved configuration:')
print(yaml.safe_dump(runtime_config, sort_keys=False))


ALGORITHM AND RUN CONFIGURATION


{
  "normalization": "Unicode NFKC + casefold; no English translation",
  "candidate_retrieval": "exact blocks + character n-gram TF-IDF + cross-script E5 reranking",
  "semantic_model": "intfloat/multilingual-e5-small",
  "classifier": "XGBoost binary:logistic with histogram trees",
  "training_device": "cpu",
  "tuning_trials": 2,
  "tuning_timeout_minutes": 2.0,
  "early_stopping_rounds": 15,
  "maximum_boost_rounds": 100,
  "selection_metric": "entity-level macro F0.5",
  "time_budget_minutes": 30,
  "output_reserve_minutes": 5
}

Full resolved configuration:
project:
  seed: 42
  time_budget_minutes: 30
  output_reserve_minutes: 5
  chunksize: 2000
  feature_chunk_rows: 5000
  sample_rows_per_file: 10000
candidate_generation:
  name_top_k: 5
  address_top_k: 5
  semantic_pairs_per_query: 0
  max_candidates_per_query: 15
  max_exact_block_size: 30
  max_tfidf_features: 25000
embeddings:
  enabled: false
training:
  tuning_trials: 2
  tuning_timeout_seconds: 120
  tuning_sample_rows

## 7. Execute with live progress

The child process is unbuffered. Every pipeline message appears immediately with a timestamp. If a stage is quiet for 60 seconds, the notebook prints a heartbeat with elapsed time so the run never looks frozen.

In [6]:
def reset_runtime_directory(path: Path, allowed_parent: Path) -> None:
    resolved_path = path.resolve()
    resolved_parent = allowed_parent.resolve()
    if resolved_path == resolved_parent or resolved_parent not in resolved_path.parents:
        raise RuntimeError(f'Refusing to reset unsafe runtime path: {resolved_path}')
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def run_with_live_output(command, *, cwd: Path, env: dict, heartbeat_seconds: int = 60) -> None:
    print('Command:', ' '.join(map(str, command)), flush=True)
    started = time.monotonic()
    process = subprocess.Popen(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_queue = queue.Queue()

    def collect_output() -> None:
        assert process.stdout is not None
        for line in process.stdout:
            output_queue.put(line.rstrip())
        output_queue.put(None)

    reader = threading.Thread(target=collect_output, daemon=True)
    reader.start()
    while True:
        try:
            line = output_queue.get(timeout=heartbeat_seconds)
        except queue.Empty:
            elapsed = (time.monotonic() - started) / 60
            print(f'[{datetime.now():%H:%M:%S}] [heartbeat] still running | elapsed={elapsed:.1f} min', flush=True)
            continue
        if line is None:
            break
        print(f'[{datetime.now():%H:%M:%S}] {line}', flush=True)

    return_code = process.wait()
    elapsed = (time.monotonic() - started) / 60
    print(f'[process] exit_code={return_code} | elapsed={elapsed:.1f} min', flush=True)
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

pipeline_env = os.environ.copy()
pipeline_env['PYTHONUNBUFFERED'] = '1'
pipeline_env['PYTHONPATH'] = str(PACKAGE_ROOT / 'src') + os.pathsep + pipeline_env.get('PYTHONPATH', '')
base_command = [
    sys.executable, '-m', 'ber.cli',
    '--config', str(RUNTIME_CONFIG),
    '--repo-root', str(PROJECT_DIR),
    '--data-root', str(DATA_ROOT),
    '--work-dir', str(WORK_DIR),
    '--output-dir', str(OUTPUT_DIR),
]

banner('PIPELINE INSPECTION')
run_with_live_output(
    [*base_command[:3], 'inspect', *base_command[3:]],
    cwd=PACKAGE_ROOT,
    env=pipeline_env,
)

if RUN_FULL_PIPELINE:
    banner('RESET RUN ARTIFACTS')
    reset_runtime_directory(WORK_DIR, Path('/kaggle/temp'))
    reset_runtime_directory(OUTPUT_DIR, Path('/kaggle/working'))

    banner('FULL PIPELINE STARTED')
    run_with_live_output(
        [*base_command[:3], 'run-all', *base_command[3:]],
        cwd=PACKAGE_ROOT,
        env=pipeline_env,
    )
else:
    print('\nLocal validation mode: setup and inspect succeeded; full training was intentionally skipped.')


PIPELINE INSPECTION


Command: C:\Python314\python.exe -m ber.cli inspect --config C:\Users\satya\AppData\Local\Temp\ber-notebook-6bu35ix1\ber_runtime_config.yaml --repo-root D:\Bunker\BaseCamp\AmazonML --data-root D:\Bunker\BaseCamp\AmazonML\dataset --work-dir C:\Users\satya\AppData\Local\Temp\ber-notebook-6bu35ix1\work --output-dir C:\Users\satya\AppData\Local\Temp\ber-notebook-6bu35ix1\output


[01:06:43] {


[01:06:43]   "config": {


[01:06:43]     "_config_hash": "e19073c87b33",


...

[01:06:43]   },


[01:06:43]   "work_dir": "C:\\Users\\satya\\AppData\\Local\\Temp\\ber-notebook-6bu35ix1\\work"


[01:06:43] }


[process] exit_code=0 | elapsed=0.3 min



Local validation mode: setup and inspect succeeded; full training was intentionally skipped.


## 8. Report the measured F0.5 and selected model

This section reads the artifacts produced by the actual run. It does not contain placeholder scores. Validation F0.5 is used during model selection; holdout F0.5 is the cleaner final estimate because the holdout entities were not used to choose hyperparameters or threshold.

In [7]:
if RUN_FULL_PIPELINE:
    completed_run = json.loads((WORK_DIR / 'completed_run.json').read_text(encoding='utf-8'))
    model_metadata = json.loads((WORK_DIR / 'model' / 'model_metadata.json').read_text(encoding='utf-8'))
    training_result = completed_run['training']

    banner('FINAL MODEL METRICS')
    print(f"Algorithm               : XGBoost binary classifier")
    print(f"Validation macro F0.5   : {training_result['validation_macro_f0_5']:.6f}")
    holdout_score = training_result['holdout_macro_f0_5']
    print(f"Holdout macro F0.5      : {holdout_score:.6f}" if holdout_score is not None else 'Holdout macro F0.5      : n/a')
    print(f"Decision threshold      : {training_result['threshold']:.3f}")
    print(f"Best boosting iteration : {model_metadata['final_best_iteration']:,}")
    print(f"Tuning search time      : {model_metadata['search_elapsed_seconds'] / 60:.2f} min")
    print(f"Total pipeline runtime  : {completed_run['elapsed_minutes']:.2f} min")

    print('\nSelected XGBoost parameters:')
    print(json.dumps(model_metadata['params'], indent=2, sort_keys=True))
else:
    print('Metrics appear here after the full Kaggle run.')

Metrics appear here after the full Kaggle run.


## 9. Compare all hyperparameter trials

Each row is a completed XGBoost trial. The table is sorted by validation macro F0.5, so the selected trial appears first.

In [8]:
if RUN_FULL_PIPELINE:
    trial_rows = []
    for trial in model_metadata['search_trials']:
        params = trial['params']
        trial_rows.append({
            'trial': trial['trial'] + 1,
            'validation_macro_F0.5': trial['macro_f0_5'],
            'threshold': trial['threshold'],
            'best_iteration': trial['best_iteration'],
            'max_depth': params['max_depth'],
            'eta': params['eta'],
            'min_child_weight': params['min_child_weight'],
            'subsample': params['subsample'],
            'colsample_bytree': params['colsample_bytree'],
            'reg_alpha': params['reg_alpha'],
            'reg_lambda': params['reg_lambda'],
        })
    trials_table = pd.DataFrame(trial_rows).sort_values(
        'validation_macro_F0.5', ascending=False
    ).reset_index(drop=True)
    display(trials_table.style.format({
        'validation_macro_F0.5': '{:.6f}',
        'threshold': '{:.3f}',
        'eta': '{:.5f}',
        'min_child_weight': '{:.3f}',
        'subsample': '{:.3f}',
        'colsample_bytree': '{:.3f}',
        'reg_alpha': '{:.5f}',
        'reg_lambda': '{:.3f}',
    }))
else:
    print('The tuning comparison appears here after training.')

The tuning comparison appears here after training.


## 10. Inspect threshold selection

The classifier produces probabilities. This curve shows why the final probability threshold was selected: it is the tested threshold with the strongest validation macro F0.5.

In [9]:
if RUN_FULL_PIPELINE:
    import matplotlib.pyplot as plt

    threshold_curve = pd.DataFrame(model_metadata['threshold_curve'])
    best_index = threshold_curve['macro_f0_5'].idxmax()
    best_point = threshold_curve.loc[best_index]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(threshold_curve['threshold'], threshold_curve['macro_f0_5'], linewidth=2, color='#2563eb')
    ax.scatter([best_point['threshold']], [best_point['macro_f0_5']], color='#dc2626', s=70, zorder=3)
    ax.annotate(
        f"best = {best_point['macro_f0_5']:.4f} @ {best_point['threshold']:.3f}",
        (best_point['threshold'], best_point['macro_f0_5']),
        xytext=(12, 12),
        textcoords='offset points',
    )
    ax.set_title('Validation macro F0.5 by decision threshold')
    ax.set_xlabel('Decision threshold')
    ax.set_ylabel('Entity-level macro F0.5')
    ax.grid(alpha=0.25)
    plt.show()
else:
    print('The threshold-selection curve appears here after training.')

The threshold-selection curve appears here after training.


## 11. Candidate recall and preprocessing diagnostics

Candidate recall answers a critical question: did candidate generation place the true match in front of the classifier? If recall is weak, tuning XGBoost cannot recover matches it never receives.

In [10]:
if RUN_FULL_PIPELINE:
    preprocessing_manifest = json.loads(
        (WORK_DIR / 'preprocessed' / 'preprocessing_manifest.json').read_text(encoding='utf-8')
    )
    preprocessing_rows = []
    for source_key, summary in preprocessing_manifest['sources'].items():
        preprocessing_rows.append({
            'source': source_key,
            'rows': summary['rows'],
            'countries': len(summary['countries']),
            'empty_addresses': summary['empty_addresses'],
            'sampled': summary['sampled'],
        })
    banner('PREPROCESSING SUMMARY')
    display(pd.DataFrame(preprocessing_rows))

    train_candidate_manifest = json.loads(
        (WORK_DIR / 'candidates' / 'train' / 'manifest.json').read_text(encoding='utf-8')
    )
    candidate_table = pd.DataFrame(train_candidate_manifest['countries'])
    banner('TRAIN CANDIDATE DIAGNOSTICS')
    display(candidate_table)
else:
    print('Preprocessing and candidate diagnostics appear here after the full run.')

Preprocessing and candidate diagnostics appear here after the full run.


## 12. Validate, preview, and package final outputs

The official validator already ran inside the pipeline. This cell shows bounded previews, preserves the model and run evidence, and creates the submission ZIP under `/kaggle/working`.

In [11]:
if RUN_FULL_PIPELINE:
    matching_path = OUTPUT_DIR / 'matching_results.tsv'
    candidate_path = OUTPUT_DIR / 'candidate_pairs.tsv'

    banner('OUTPUT FILES')
    for path in [matching_path, candidate_path]:
        print(f'{path.name:24s} {path.stat().st_size / (1024**2):10.2f} MiB | {path}')

    print('\nmatching_results.tsv preview:')
    display(pd.read_csv(matching_path, sep='\t', dtype=str, keep_default_na=False, nrows=5))
    print('\ncandidate_pairs.tsv preview:')
    display(pd.read_csv(candidate_path, sep='\t', dtype=str, keep_default_na=False, nrows=5))

    package_root = Path('/kaggle/working/package')
    if package_root.exists():
        shutil.rmtree(package_root)
    shutil.copytree(OUTPUT_DIR, package_root / 'output')
    shutil.copytree(
        PACKAGE_ROOT,
        package_root / 'code' / 'business_entity_resolution',
        ignore=shutil.ignore_patterns('__pycache__', '*.pyc', 'artifacts'),
    )
    shutil.copy2(PROJECT_DIR / 'Documentation_template.md', package_root / 'Documentation_template.md')

    evidence_dir = Path('/kaggle/working/run_evidence')
    evidence_dir.mkdir(parents=True, exist_ok=True)
    for path in [WORK_DIR / 'run_metadata.json', WORK_DIR / 'completed_run.json', RUNTIME_CONFIG]:
        shutil.copy2(path, evidence_dir / path.name)
    shutil.copytree(WORK_DIR / 'model', evidence_dir / 'model', dirs_exist_ok=True)
    shutil.copytree(WORK_DIR / 'candidates', evidence_dir / 'candidate_manifests', dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('*.parquet'))

    archive_base = Path('/kaggle/working') / f'{TEAM_NAME}_submission'
    archive_path = Path(shutil.make_archive(str(archive_base), 'zip', package_root))
    print(f'\nSubmission archive: {archive_path} ({archive_path.stat().st_size / (1024**2):.2f} MiB)')
    print('Save a Kaggle notebook version now so /kaggle/working outputs persist.')
else:
    print('Packaging runs after the full Kaggle pipeline completes.')

Packaging runs after the full Kaggle pipeline completes.


## 13. How to read the result

- Trust the **untouched holdout macro F0.5** more than the best tuning-trial score.
- If candidate recall is low, improve blocking and retrieval first.
- If candidate recall is high but F0.5 is low, improve pair features, XGBoost tuning, or threshold calibration.
- A completed run does not guarantee 0.99. Only the printed validation and holdout measurements establish model quality.
- The Git commit, configuration, hardware, model, metrics, and output files are saved so the experiment can be reproduced.